In [ ]:
/*
========================================================
FAILURE DETECTION AND HEARTBEATS - SIMPLE HINGLISH NOTES
========================================================

INTRO
--------------------------------------------------------
Distributed systems mein nodes hamesha fail hote rehte 
hain. Servers crash hote, network partition hota, processes 
hang ho jaate. Sawal ye nahi ki "failure hogi ya nahi", 
sawal ye hai "kitni JALDI detect kar paoge?"

EXAMPLE: 3 replicas wala database cluster. Primary node 
crash ho jata. Agar detect karne mein 5 MINUTE lage, to 5 
minute downtime. Agar 5 SECOND mein detect ho jaye, to sirf 
5 second downtime.

Failure detection fault tolerance ki foundation hai. Iske 
bina, failover trigger nahi kar sakte, load rebalance nahi 
kar sakte, operators ko alert nahi kar sakte.

TEXT DIAGRAM:

  Node A (💓 heartbeat) --\
  Node B (💓 heartbeat) --- [Monitor]
  Node C (❌ no heartbeat)-/
       |
  "Node C failed!" -> Trigger Failover


KAHA KAHA YE PATTERN USE HOTA
--------------------------------------------------------
  Problem                  Kaise Use Hota
  ---------------------------------------------------------
  Distributed Database     Failed replicas detect karna, 
                             leader election trigger karna
  Load Balancer            Unhealthy servers rotation se 
                             remove karna
  Message Queue            Dead consumers detect karna, 
                             partitions reassign karna
  Distributed Cache        Failed nodes detect karna, data 
                             rebalance karna
  Service Discovery        Services ko unhealthy mark karna
  Coordination Service     Leader election, distributed 
                             locking
  Container Orchestration  Failed containers restart karna
  Chat/Messaging           Offline users detect karna, 
                             presence indicators


1. FAILURE DETECTION KYA HAI?
--------------------------------------------------------
Failure detection wo mechanism hai jisse distributed system 
ke nodes pata karte hain ki dusre nodes zinda hain ya mare 
hue.

TEXT DIAGRAM - SIMPLE HEALTH CHECK:

  Node A -> "Are you alive?" -> Node B
  Node B -> "Yes, I'm here!" -> Node A

Simple lagta hai, na? Bas pucho aur wait karo. Lekin YAHI 
distributed systems mein tricky ban jata hai.

AGAR Node B RESPOND NAHI KARE, TO KYA HUA?
- Crashed hai? (process mar gaya)
- Slow hai? (overloaded)
- Network issue hai? (message lost ho gaya)
- Partitioned hai? (network down)

TEXT DIAGRAM:

  No Response from Node B
       |
  ----------------------------------------
  | Crashed?  | Slow?     | Network     | Partitioned? |
  | Process   | Overloaded| Issue?      | Network down |
  | died      |           | Msg lost    |              |
  ----------------------------------------
       |
  "YOU CANNOT TELL WHICH ONE!"

BADI BAAT: Distributed systems mein response na milna 
"FAILURE" nahi hai, ye "UNKNOWN" hai. Isiliye failure 
detectors "SUSPICION" dete hain, "CERTAINTY" nahi. Ek node 
tab tak "SUSPECTED" rehta hai jab tak proven na ho.


2. FAILURE DETECTION HARD KYU HAI?
--------------------------------------------------------

--------------------------------------------------------
2.1 THE TWO GENERALS PROBLEM
--------------------------------------------------------
Do generals attack coordinate karna chahte, but sirf 
messengers se baat kar sakte jo capture ho sakte. Koi bhi 
general kabhi CERTAIN nahi ho sakta ki dusre ko message 
mila.

TEXT DIAGRAM:

  General A -> "Attack at dawn!" (Messenger 1) -> General B
     "Might be captured..."
  General B -> "Confirmed!" (Messenger 2) -> General A
     "Might also be captured..."
     
  A sochta: "Kya main sure hoon B ko mila?"
  B sochta: "Kya main sure hoon A ko confirmation mila?"

FAILURE DETECTION MEIN YE KAISE APPLY HOTA:
Jab Node A, Node B ko heartbeat request bhejta aur response 
nahi aata, A ko pata nahi:
- B ko request mili hi nahi
- B ko mili but response lost ho gaya
- B mar gaya

--------------------------------------------------------
2.2 ASYNCHRONOUS NETWORKS
--------------------------------------------------------
Real networks mein unpredictable delays hote hain. Message 
1ms mein bhi aa sakta, 1 second mein bhi. Koi upper bound 
nahi ki message kitni der mein aayega.

TEXT DIAGRAM:

  NORMAL CASE:       Node A -> 10ms -> Node B
  CONGESTED NETWORK: Node A -> 500ms -> Node B
  PACKET LOSS+RETRY: Node A -> lost ❌ -> retry after 2s -> Node B

Bina guaranteed upper bound ke, kabhi CERTAIN nahi ho sakte 
ki node dead hai. Ho sakta wo bas high latency face kar raha 
ho.

--------------------------------------------------------
2.3 FUNDAMENTAL TRADE-OFF
--------------------------------------------------------
Failure detection mein 2 competing goals ke beech choose 
karna padta:

  Fast Detection  -> Low timeout, fast failover, but MORE 
                      false positives
  Avoid False Pos -> High timeout, kam mistakes, but 
                      SLOWER failover

TEXT DIAGRAM:

  TIMEOUT TOO LONG (30s):
    Actually Dead Node -> No response for 30s -> Still 
    marked ALIVE -> SLOW DETECTION!

  TIMEOUT TOO SHORT (100ms):
    Healthy but Slow Node -> No response within 100ms -> 
    Marked DEAD -> FALSE POSITIVE!

Koi perfect answer nahi hai. Har system ko line kahan 
khinchni hai, decide karna padta hai.


3. HEARTBEATS KAISE KAAM KARTE HAIN
--------------------------------------------------------
Heartbeat ek periodic message hai jo liveness batane ke 
liye bheja jaata. Simple idea: agar node se sunte rehte ho, 
to wo alive hai. Sunna band ho jaye, to shayad dead hai.

--------------------------------------------------------
3.1 PUSH vs PULL MODEL
--------------------------------------------------------

PUSH MODEL: Har node periodically "I'm alive" broadcast 
karta.

TEXT DIAGRAM:

  Node A (💓 every 1s) --\
  Node B (💓 every 1s) ---> [Monitor]
  Node C (💓 every 1s) --/

PULL MODEL: Ek central monitor periodically har node se 
pucchta "Are you alive?"

TEXT DIAGRAM:

  [Monitor] -- ping --> Node A -- pong --> [Monitor]
  [Monitor] -- ping --> Node B -- pong --> [Monitor]
  [Monitor] -- ping --> Node C -- pong --> [Monitor]
  (Every 1 second loop)

HYBRID MODEL: Nodes normally push heartbeats, monitor sirf 
pull karta jab kaafi time se kuch sunai nahi diya.

  Model   Pros                          Cons
  ---------------------------------------------------------
  Push    Kam latency detection, monitor Zyada network 
          pe kam load                     traffic
  Pull    Centralized control, manage    Monitor bottleneck 
          karna easy                      ban sakta
  Hybrid  Dono ka best, adaptive         Zyada complex

--------------------------------------------------------
3.2 HEARTBEAT MEIN KYA HOTA HAI?
--------------------------------------------------------
Heartbeat simple "I'm alive" ho sakta, ya extra info bhi 
carry kar sakta - jaise CPU load, memory usage, queue 
depth, version number.

Rich heartbeats se sophisticated decisions liye ja sakte. 
Load balancer sirf "alive hai" nahi, "overloaded hai" bhi 
check kar sakta.

--------------------------------------------------------
3.3 HEARTBEAT INTERVAL AND TIMEOUT
--------------------------------------------------------
2 CRITICAL PARAMETERS:

  Heartbeat Interval -> Kitni baar heartbeat bhejta hai 
                         (e.g., every 1 second)
  Failure Timeout    -> Kitni der wait karein bina 
                         heartbeat ke, failure suspect karne 
                         se pehle (e.g., 5 seconds)

TEXT DIAGRAM - Timeline (1s interval, 3s timeout):

  t=0: Heartbeat received
  t=1: Heartbeat received
  t=2: Heartbeat received
  t=3: No heartbeat (miss 1)
  t=4: No heartbeat (miss 2)
  t=5: No heartbeat (miss 3)
  t=6: SUSPECTED!

RELATIONSHIP - Interval aur Timeout ka ratio:

  1x Interval  -> Timeout = 1s  -> Fast detection, MANY 
                                    false positives
  3x Interval  -> Timeout = 3s  -> Good balance 
                                    (RECOMMENDED)
  10x Interval -> Timeout = 10s -> Slow detection, FEW 
                                    false positives


4. FAILURE DETECTION STRATEGIES
--------------------------------------------------------

--------------------------------------------------------
4.1 FIXED TIMEOUT
--------------------------------------------------------
Sabse simple: agar fixed time ke andar heartbeat nahi aaya, 
to node ko FAILED mark kardo.

TEXT DIAGRAM:

  Config: interval=1s, timeout=5s
  Monitor <- heartbeat (t=0) -- Node    | Last seen: t=0
  Monitor <- no heartbeat (t=1) -- Node
  Monitor <- no heartbeat (t=2) -- Node
  Monitor <- no heartbeat (t=3) -- Node
  Monitor <- no heartbeat (t=4) -- Node
  t=5: Timeout exceeded! -> Mark node as FAILED

ACHA (PROS):
- Implement karna simple
- Predictable behavior
- Reason karna easy

BURA (CONS):
- Network conditions ke hisab se adapt nahi karta
- Fast aur slow network dono ke liye same timeout
- Network hiccups mein false positives deta

--------------------------------------------------------
4.2 ADAPTIVE TIMEOUT
--------------------------------------------------------
Observed network conditions ke basis pe timeout adjust 
karta. Agar heartbeats normally 50ms mein aate, to timeout 
kaafi kam rakho (agar 500ms mein aate to zyada rakho).

TEXT DIAGRAM:

  Observed Heartbeat Delays: 48ms, 52ms, 47ms, 55ms, 51ms
       |
  Calculate Statistics:
    Mean = 50.6ms
    Std Dev = 3.2ms
       |
  Timeout = mean + 4×stddev
       |
  Adaptive Timeout = 63.4ms

KEY IDEA: Heartbeat arrival times ki history track karo, 
statistics use karke aisa timeout set karo jo normal 
variation account kare.

ACHA (PROS):
- Network conditions ke hisab se adapt karta
- Kam false positives
- Different environments mein kaam karta

BURA (CONS):
- Implement karna zyada complex
- Data collect karne ke liye warm-up period chahiye
- Sudden changes ko adapt karne mein slow ho sakta

--------------------------------------------------------
4.3 PHI ACCRUAL FAILURE DETECTOR
--------------------------------------------------------
Cassandra aur Akka use karte. Ye binary alive/dead decision 
nahi deta. Iske bajaye, ek "suspicion level" (phi) output 
karta jo time ke saath badhta hai.

TEXT DIAGRAM - Phi Interpretation:

  φ = 1  -> 10% wrong probability   -> Probably alive
  φ = 2  -> 1% wrong probability    -> Possibly dead
  φ = 4  -> 0.01% wrong probability -> Likely dead
  φ = 8  -> 0.000001% wrong         -> Definitely dead

Applications ek threshold set karte (commonly φ = 8). Jab 
phi threshold cross kare, node ko dead maan liya jaata.

KAISE KAAM KARTA:
1. Heartbeats ke arrival times track karo
2. Inter-arrival times ki distribution model karo (usually 
   exponential)
3. Probability calculate karo ki next heartbeat late hoga
4. Probability ko phi mein convert karo: φ = -log10(probability)

ACHA (PROS):
- Uncertainty quantification deta
- Network conditions ke hisab se automatically adapt karta
- Applications apna risk tolerance set kar sakte

BURA (CONS):
- Math zyada complex hai
- Threshold tune karna padta
- Assume karta ki heartbeat arrivals predictable distribution 
  follow karte

--------------------------------------------------------
4.4 GOSSIP-BASED FAILURE DETECTION
--------------------------------------------------------
Central monitor ki jagah, nodes ek dusre ki health ke baare 
mein "gossip" karte. Har node ek view maintain karta sab 
nodes ka, aur gossip ke dauran share karta.

TEXT DIAGRAM - Gossip Exchange:

  Node A's View:            Node B's View:
  Node | HB  | Last Update  Node | HB  | Last Update
  A    | 150 | now          A    | 147 | 2s ago
  B    | 148 | 1s ago       B    | 149 | now
  C    | 145 | 3s ago       C    | 146 | 2s ago
  D    | 120 | 30s ago ⚠️   D    | 121 | 28s ago ⚠️
       |                          |
       -------- Merge Views --------
             (max heartbeat wins)
       |
  "Both nodes now have latest info"

TEXT DIAGRAM - Gossip Spreading Failure Info:

  Node D crashes!
       |
  Node A -- gossip(D: hb=120, 5s ago) --> Node B
  Node B -- gossip(D: hb=120, 8s ago) --> Node C
       |
  "All nodes now suspect D" (30s timeout)

Har node apna heartbeat counter periodically increment 
karta. Gossip ke dauran, nodes apne views exchange aur merge 
karte.

ACHA (PROS):
- Decentralized, single point of failure nahi
- Bade clusters mein achha scale karta
- Central coordination ke bina info phailti

BURA (CONS):
- Eventually consistent (turant detection nahi)
- Implement karna zyada complex
- Detection time gossip frequency pe depend karta


5. TRADE-OFFS JO DHYAN MEIN RAKHNE
--------------------------------------------------------
Interview mein ye samajhna zaroori hai - koi perfect 
solution nahi hai.

--------------------------------------------------------
5.1 DETECTION TIME vs FALSE POSITIVES
--------------------------------------------------------
Fast detection AUR zero false positives dono ek saath 
NAHI ho sakte.

AGGRESSIVE SETTINGS (low timeout):
- Fast failover
- Quick recovery
- BUT: GC pauses/network blips mein nodes dead mark ho 
  jaate

CONSERVATIVE SETTINGS (high timeout):
- Accurate detection
- Kam unnecessary failovers
- BUT: Actual failure hone pe lambi downtime

--------------------------------------------------------
5.2 FALSE POSITIVES KA COST
--------------------------------------------------------
False positive (healthy node ko dead mark karna) 
unnecessary actions trigger karta:
- Load balancer working server remove kar deta
- Cluster unnecessarily data rebalance karta
- Naya leader elect ho jata jab purana leader kaam kar raha 
  ho (SPLIT-BRAIN!)
- Unnecessary alerts se on-call engineers uth jaate

Jaha ye actions expensive hain (jaise data rebalancing), 
conservative timeouts prefer kiye jaate.

--------------------------------------------------------
5.3 SLOW DETECTION KA COST
--------------------------------------------------------
Slow detection ka matlab:
- Zyada downtime
- Users errors experience karte
- Failover tak data unavailable
- SLA violations

User-facing systems mein jaha availability critical hai, 
kuch false positives sahke bhi FASTER detection worth karta 
hai.

--------------------------------------------------------
5.4 RIGHT TIMEOUT CHOOSE KARNA
--------------------------------------------------------
Timeout value mein system ke saare delay sources account 
hone chahiye.

TEXT DIAGRAM:

  Delay Sources:
    Network Latency   (p99: 50ms)
    GC Pause          (max: 500ms)
    Heartbeat Interval (1000ms)
         |
    Sum: 1550ms
         |
    Add Safety Margin (2x)
         |
    Final Timeout: 3000ms

DHYAN MEIN RAKHO YE FACTORS:
- Network latency (p99 use karo, AVERAGE nahi!)
- GC pause times
- Disk I/O latency
- Apne system mein false positives ka cost
- Required availability SLA


6. REAL-WORLD IMPLEMENTATIONS
--------------------------------------------------------

--------------------------------------------------------
6.1 APACHE CASSANDRA
--------------------------------------------------------
Cassandra Phi Accrual Failure Detector + Gossip use karta.

TEXT DIAGRAM:

  Gossip Protocol (Every 1 second)
       |
  Phi Accrual Detector (Threshold: φ = 8)
       |
  φ < 8  -> Node ALIVE
  φ ≥ 8  -> Node DOWN -> Excluded from read/write operations

KAISE KAAM KARTA:
1. Nodes har second gossip karte, heartbeat info exchange 
   karte
2. Har node har dusre node ke liye phi calculate karta
3. Jab phi 8 se zyada ho jaye, node DOWN mark hota
4. Marked-down nodes read/write operations se exclude ho 
   jaate

Cassandra ne phi accrual isliye choose kiya kyunki ye 
different data centers ke network conditions ke hisab se 
automatically adapt karta hai.

--------------------------------------------------------
6.2 APACHE ZOOKEEPER
--------------------------------------------------------
ZooKeeper session-based failure detection use karta.

TEXT DIAGRAM:

  Client -> Create session (timeout=10s) -> ZooKeeper
  ZooKeeper -> Session established -> Client
  [Loop every 3 seconds: heartbeat -> ack]
  
  Client crashes!
       |
  No heartbeat for 10s -> Session expired!
       |
  Delete ephemeral znodes -> Leader election triggered

KAISE KAAM KARTA:
1. Clients ZooKeeper servers ke saath sessions maintain 
   karte
2. Sessions ka timeout hota (default: 6-40 seconds)
3. Clients session timeout ke 1/3 time ke andar heartbeat 
   bhejte
4. Session timeout ke andar heartbeat na aaye to session 
   expire ho jata
5. Us client ke banaye ephemeral nodes delete ho jaate

Ye leader election ke liye use hota: leader ek ephemeral 
node banata, agar wo mar jaye, node gayab ho jata, jisse 
naya election trigger hota.

--------------------------------------------------------
6.3 KUBERNETES
--------------------------------------------------------
Kubernetes multiple layers ka failure detection use karta - 
"defense in depth".

TEXT DIAGRAM:

  POD LEVEL:
    Liveness Probe (3 failures) -> Restart Container
    Readiness Probe (failures)  -> Remove from Service
  
  NODE LEVEL:
    Kubelet Heartbeat (Every 10s) -> 40s timeout -> Node 
    NotReady

PROBE TYPES:
- Liveness Probe: Container zinda hai? Nahi to restart karo.
- Readiness Probe: Container traffic serve kar sakta? Nahi 
  to load balancer se remove karo.

--------------------------------------------------------
6.4 CONSUL
--------------------------------------------------------
HashiCorp Consul SWIM protocol use karta (Scalable 
Weakly-consistent Infection-style Process Group Membership).

TEXT DIAGRAM:

  Node A -- ping --> Node D -- no response
       |
  Node A -- "ping D for me" --> Node B, Node C
  Node B -- ping --> Node D -- no response
  Node C -- ping --> Node D -- no response
       |
  "D didn't respond" (from B and C too)
       |
  Mark D as SUSPECTED (after 5s grace period)
       |
  Mark D as FAILED

SWIM KAISE KAAM KARTA:
1. Har node periodically ek random peer ko ping karta
2. Response na aaye to, dusre nodes se puchta us suspect ko 
   ping karne ke liye (indirect probe)
3. Wo bhi response na paayein to, SUSPECTED mark hota
4. Grace period ke baad, FAILED mark hota

SWIM ka indirect probing false positives kam karta jo 
specific node-pairs ke beech network issues se aate. Agar A, 
D tak nahi pahunch pa raha but B aur C pahunch sakte, to D 
galat tarike se dead mark nahi hoga.

--------------------------------------------------------
6.5 COMPARISON TABLE
--------------------------------------------------------
  System      Detection Method       Timeout          Architecture
  -----------------------------------------------------------------
  Cassandra   Phi Accrual + Gossip   Adaptive (φ=8)   Decentralized
  ZooKeeper   Session-based          6-40s            Centralized
  Kubernetes  Multi-layer probes     40s (node)       Hierarchical
  Consul      SWIM protocol          Configurable     Decentralized


7. BEST PRACTICES
--------------------------------------------------------

--------------------------------------------------------
7.1 MULTIPLE HEALTH SIGNALS USE KARO
--------------------------------------------------------
Sirf heartbeats pe depend mat karo. Multiple signals 
combine karo.

TEXT DIAGRAM:

  Health Signals:
    Heartbeat (Liveness)      -> Zinda hai?
    Can serve requests?       -> Readiness
    What's the load?          -> Capacity
    Any errors?               -> Quality
         |
    All signals positive? -> YES -> HEALTHY
                           -> NO  -> UNHEALTHY

--------------------------------------------------------
7.2 GRACEFUL DEGRADATION IMPLEMENT KARO
--------------------------------------------------------
Node ko FAILED mark karne se pehle, kam drastic measures 
try karo.

TEXT DIAGRAM:

  Slow responses -> Reduce traffic
  Some failures  -> Stop new requests
  All failures   -> Remove from rotation
  No heartbeat   -> Mark as failed
  Prolonged failure -> Trigger replacement -> Node replaced

--------------------------------------------------------
7.3 NETWORK PARTITIONS HANDLE KARO
--------------------------------------------------------
Network partition healthy nodes ko dead dikha sakta. Isko 
design mein consider karo.

TEXT DIAGRAM:

  [Partition A]              [Partition B]
  Node 1, Node 2  <--❌-->   Node 3, Node 4
                Network Partition
  
  "A thinks B is dead"
  "B thinks A is dead"

SOLUTION: Quorum-based decisions use karo
- Failure declare karne se pehle majority agreement 
  zaroori
- Dono partitions ko independently action lene se roko 
  (split-brain prevent)

--------------------------------------------------------
7.4 LOG AND ALERT ON FAILURE DETECTION EVENTS
--------------------------------------------------------
Failure detection ko observable banao:
- Har state transition log karo (HEALTHY -> SUSPECTED -> 
  FAILED)
- Detection times track karo
- Frequent state changes pe alert karo (flapping)
- Cluster health ka dashboard banao

--------------------------------------------------------
7.5 FAILURE DETECTION TEST KARO
--------------------------------------------------------
Regularly verify karo ki failure detection kaam kar raha:
- Chaos testing: Nodes kill karo, detection time measure 
  karo
- Network partition testing: Nodes isolate karo, behavior 
  verify karo
- Slow node testing: Latency inject karo, false positives 
  check karo
- GC pause simulation: Verify karo pauses ke dauran nodes 
  dead mark nahi ho rahe


8. REFERENCES (Padhne ke liye)
--------------------------------------------------------
- The Phi Accrual Failure Detector - Hayashibara et al. ka 
  original paper
- SWIM Protocol - Consul aur Serf ke peeche ka protocol
- Cassandra Failure Detection Documentation
- Designing Data-Intensive Applications - Chapter 8 
  (unreliable networks aur failure detection)
- ZooKeeper Internals - Session management aur heartbeats


9. KEY TAKEAWAYS
--------------------------------------------------------
- Failure detection mein "response na aana" = FAILURE nahi, 
  UNKNOWN hai
- Fundamental trade-off: Fast detection vs False positives 
  - dono ek saath nahi mil sakte
- Fixed timeout simple hai but adapt nahi karta
- Adaptive timeout network conditions ke hisab se adjust 
  hota
- Phi Accrual Failure Detector suspicion LEVEL deta, binary 
  decision nahi (Cassandra/Akka use karte)
- Gossip-based detection decentralized hai, koi single point 
  of failure nahi
- Network partitions handle karne ke liye quorum-based 
  decisions use karo (split-brain avoid karne ke liye)
- Multiple health signals combine karo, sirf heartbeat pe 
  depend mat karo
- Timeout choose karte waqt p99 latency, GC pauses, disk I/O 
  sab account karo

========================================================
END OF NOTES
========================================================
*/